In [0]:
dbutils.widgets.removeAll() 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("source", "silver")
dbutils.widgets.text("sink", "golden")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
schema_source = dbutils.widgets.get("source")
schema_sink = dbutils.widgets.get("sink")

## 1. Lectura de tabla fuente Silver

In [0]:
df_silver = spark.table(f"{catalogo}.{schema_source}.ventas_productos_categorias")
display(df_silver)

## 2. Transformación: agregación mensual por categoría

In [0]:
df_ventas_categoria = (
    df_silver
    .withColumn("ANIO", F.year(F.col("FECHA")))
    .withColumn("MES",  F.month(F.col("FECHA")))
    .groupBy(
        F.col("CATEGORIA"),
        F.col("ANIO"),
        F.col("MES")
    )
    .agg(
        # --- Volumen ---
        F.countDistinct("ID_PEDIDO").alias("NUM_PEDIDOS"),
        F.countDistinct("ID_ARTICULO").alias("NUM_ARTICULOS"),
        F.sum("CANTIDAD").alias("TOTAL_UNIDADES"),
        F.count("*").alias("NUM_LINEAS_TOTAL"),

        # --- Ingresos ---
        F.round(
            F.sum(F.col("PRECIO") * F.col("CANTIDAD")), 2
        ).alias("INGRESO_BRUTO"),
        F.round(
            F.sum(
                F.col("PRECIO") * F.col("CANTIDAD") * (1 - F.col("DESCUENTO") / 100)
            ), 2
        ).alias("INGRESO_NETO"),
        F.round(
            F.avg("PRECIO"), 2
        ).alias("PRECIO_PROMEDIO"),

        # --- Descuento ---
        F.count(
            F.when(F.col("DESCUENTO") > 0, 1)
        ).alias("NUM_LINEAS_CON_DESCUENTO"),
        F.round(
            F.avg(
                F.when(F.col("DESCUENTO") > 0, F.col("DESCUENTO"))
            ), 2
        ).alias("DESCUENTO_MEDIO_PCT"),

        # --- Etiquetas de descuento ---
        F.count(
            F.when(F.col("ETIQUETA_DESCUENTO") == "Precio Normal", 1)
        ).alias("NUM_PRECIO_NORMAL"),
        F.count(
            F.when(F.col("ETIQUETA_DESCUENTO") == "promoción", 1)
        ).alias("NUM_PROMOCION"),
        F.count(
            F.when(F.col("ETIQUETA_DESCUENTO") == "bono empresarial", 1)
        ).alias("NUM_BONO_EMPRESARIAL"),
        F.count(
            F.when(F.col("ETIQUETA_DESCUENTO") == "Regalo", 1)
        ).alias("NUM_REGALO")
    )
    .withColumn(
        "TICKET_PROMEDIO",
        F.round(F.col("INGRESO_NETO") / F.col("NUM_PEDIDOS"), 2)
    )
    .withColumn(
        "PCT_LINEAS_CON_DESCUENTO",
        F.round(
            F.col("NUM_LINEAS_CON_DESCUENTO") / F.col("NUM_LINEAS_TOTAL") * 100, 2
        )
    )
    .withColumn(
        "FECHA_ACTUALIZACION",
        F.current_timestamp()
    )
)

display(df_ventas_categoria)

## 3. Cast final y escritura en Gold

In [0]:
target_table = f"{catalogo}.{schema_sink}.ventas_categoria_mes"

(
    df_ventas_categoria
    .select(
        F.col("CATEGORIA").cast("string").alias("CATEGORIA"),
        F.col("ANIO").cast("int").alias("ANIO"),
        F.col("MES").cast("int").alias("MES"),
        F.col("NUM_PEDIDOS").cast("int").alias("NUM_PEDIDOS"),
        F.col("NUM_ARTICULOS").cast("int").alias("NUM_ARTICULOS"),
        F.col("TOTAL_UNIDADES").cast("int").alias("TOTAL_UNIDADES"),
        F.col("NUM_LINEAS_TOTAL").cast("int").alias("NUM_LINEAS_TOTAL"),
        F.col("INGRESO_BRUTO").cast("double").alias("INGRESO_BRUTO"),
        F.col("INGRESO_NETO").cast("double").alias("INGRESO_NETO"),
        F.col("TICKET_PROMEDIO").cast("double").alias("TICKET_PROMEDIO"),
        F.col("PRECIO_PROMEDIO").cast("double").alias("PRECIO_PROMEDIO"),
        F.col("NUM_LINEAS_CON_DESCUENTO").cast("int").alias("NUM_LINEAS_CON_DESCUENTO"),
        F.col("PCT_LINEAS_CON_DESCUENTO").cast("double").alias("PCT_LINEAS_CON_DESCUENTO"),
        F.col("DESCUENTO_MEDIO_PCT").cast("double").alias("DESCUENTO_MEDIO_PCT"),
        F.col("NUM_PRECIO_NORMAL").cast("int").alias("NUM_PRECIO_NORMAL"),
        F.col("NUM_PROMOCION").cast("int").alias("NUM_PROMOCION"),
        F.col("NUM_BONO_EMPRESARIAL").cast("int").alias("NUM_BONO_EMPRESARIAL"),
        F.col("NUM_REGALO").cast("int").alias("NUM_REGALO"),
        F.col("FECHA_ACTUALIZACION").cast("timestamp").alias("FECHA_ACTUALIZACION")
    )
    .coalesce(4)
    .write
    .mode("overwrite")
    .insertInto(target_table)
)

## 4. Validación

SELECT count(*) FROM catalog_dev.golden.ventas_categoria_mes;